In [2]:
import polars as pl
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from linearmodels.iv import compare
from linearmodels.panel.results import compare
import statsmodels.api as sm
from statsmodels.api import qqplot
from linearmodels.panel import PanelOLS
from scipy import stats as scipy_stats

import pyreadstat as prs
import pygwalker as pyg

import altair as alt
import seaborn as sns
import matplotlib.pyplot as plt

import json

from tqdm.notebook import tqdm
from itables import init_notebook_mode

alt.data_transformers.enable("vegafusion")

DataTransformerRegistry.enable('vegafusion')

In [5]:
additional_health_variables = pl.read_excel('../data/disease_variables.xlsx')
additional_health_variables

var,label,fill
str,str,str
"""m20_61""","""chronic_heart""","""no"""
"""m20_62""","""chronic_lung""","""no"""
"""m20_63""","""chronic_liver""","""no"""
"""m20_64""","""chronic_kidney""","""no"""
"""m20_65""","""chronic_stomach""","""no"""
"""m20_66""","""chronic_spinal""","""no"""
"""m20_612""","""disease_neuro""","""no"""
"""m20_613""","""disease_eye""","""no"""
"""m20_615""","""allegies""","""no"""


In [6]:
cols_disease = additional_health_variables['var'].to_list()
cols = ['idind', 'year', 'h7_2', 'h7_1', 'origsm', 'region', 'psu', 'status', 'age', 'educ', 'marst', 'h5', 'm20_7', 'j1', 'j60', 'j40', 'j363']
cols + cols_disease

['idind',
 'year',
 'h7_2',
 'h7_1',
 'origsm',
 'region',
 'psu',
 'status',
 'age',
 'educ',
 'marst',
 'h5',
 'm20_7',
 'j1',
 'j60',
 'j40',
 'j363',
 'm20_61',
 'm20_62',
 'm20_63',
 'm20_64',
 'm20_65',
 'm20_66',
 'm20_612',
 'm20_613',
 'm20_615',
 'm20_8']

In [7]:
education_levels = pl.read_excel('../data/all_education_levels.xlsx', sheet_name='selected_levels')
education_levels

educ,educ_level
str,str
"""1-2 years in Institute, Univer…","""university"""
"""7 grades of school""","""school_or_less"""
"""6 grades of school""","""school_or_less"""
"""3 grades of school""","""school_or_less"""
"""7-9 grades of school [unfinish…","""common"""
…,…
"""7-9 grades of school [unfinish…","""common"""
"""Graduate school, residency wit…","""common"""
"""10 and more grades of school &…","""higher"""


In [8]:
marital_status = pl.read_excel('../data/all_marst_levels.xlsx', sheet_name='selected_levels')
marital_status

marst,is_married
str,i64
"""In a registered marriage""",1
"""Divorsed and not remarried""",0
"""Living together, not registere…",1
"""Registered, not living togethe…",1
"""Never married""",0
"""Widower or widow""",0


In [9]:
period_relevance = pl.read_excel('../data/period_relevance.xlsx').with_columns(
    pl.col('year').cast(pl.Float64)
)
period_relevance

year,period_relevance
f64,i64
2014.0,-5
2015.0,-4
2016.0,-3
2017.0,-2
2018.0,-1
…,…
2020.0,1
2021.0,2
2022.0,3


In [10]:
is_town = pl.read_excel('../data/is_town.xlsx')
is_town

status,is_town
str,i64
"""town""",1
"""oblastnoy center""",1
"""PGT""",1
"""rural""",0


In [152]:
panel = pl.read_parquet('../data/RLMS_IND_2015_2024_eng_prepared.parquet', columns=cols + cols_disease)
for var, label, fill in zip(additional_health_variables['var'], additional_health_variables['label'], additional_health_variables['fill']):
    panel = panel.with_columns(
        pl.col(var).fill_null(fill).alias(label)
    )
panel = panel.filter(
    pl.col('year').is_in([2019, 2020])
)
panel = panel.filter(
    pl.col('h7_2').is_in(['November', 'September', 'December'])
)
panel = panel.filter(
    pl.col('age') >= 18
)
# panel = panel.with_columns(
#     pl.when(pl.col('chronic_spinal') == 'Yes').then(1).otherwise(0).alias('chronic_spinal'),
#     pl.when(pl.col('disease_eye') == 'Yes').then(1).otherwise(0).alias('disease_eye'),
#     pl.when(pl.col('disease_neuro') == 'Yes').then(1).otherwise(0).alias('disease_neuro')
# )
panel

idind,year,h7_2,h7_1,origsm,region,psu,status,age,educ,marst,h5,m20_7,j1,j60,j40,j363,m20_61,m20_62,m20_63,m20_64,m20_65,m20_66,m20_612,m20_613,m20_615,m20_8,chronic_heart,chronic_lung,chronic_liver,chronic_kidney,chronic_stomach,chronic_spinal,disease_neuro,disease_eye,allegies,disability_class
f64,f64,str,f64,str,str,str,str,f64,str,str,str,str,str,f64,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
9.0,2019.0,"""November""",13.0,"""Yes, it is an adress from the …","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",73.0,"""Technical, medical, musis etc …","""In a registered marriage""","""FEMALE""","""No""","""You are not working""",16000.0,null,16000.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""","""no"""
28.0,2019.0,"""November""",3.0,"""No, it is not belong to the re…","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",51.5,"""7-9 grades of school [unfinish…","""In a registered marriage""","""MALE""","""No""","""You are not working""",null,null,null,"""No""","""No""","""Yes""","""No""","""No""","""Yes""","""No""","""No""","""Yes""",null,"""No""","""No""","""Yes""","""No""","""No""","""Yes""","""No""","""No""","""Yes""","""no"""
30.0,2019.0,"""November""",3.0,"""No, it is not belong to the re…","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",25.0,"""10 and more grades of school w…","""Never married""","""MALE""","""No""","""You are currently working""",null,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no"""
36.0,2019.0,"""November""",1.0,"""Yes, it is an adress from the …","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",65.0,"""8 grades of school""","""In a registered marriage""","""FEMALE""","""No""","""You are not working""",14700.0,null,14700.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""no"""
24104.0,2019.0,"""November""",1.0,"""No, it is not belong to the re…","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",55.5,"""Technical, medical, musis etc …","""Divorsed and not remarried""","""FEMALE""","""No""","""You are currently working""",20200.0,null,8200.0,"""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""Yes""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""Yes""","""No""","""no"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
59408.0,2020.0,"""September""",12.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""PGT""",69.0,"""Secondary School Diploma""","""Widower or widow""","""FEMALE""","""No""","""You are not working""",11660.0,null,10660.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no"""
59409.0,2020.0,"""September""",12.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""PGT""",49.0,"""10 and more grades of school &…","""Divorsed and not remarried""","""MALE""","""No""","""You are not working""",0.0,null,null,"""No""","""No""","""No""","""No""","""No""",null,"""No""",null,"""No""",null,"""No""","""No""","""No""","""No""","""No""","""no""","""No""","""no""","""No""","""no"""
59411.0,2020.0,"""November""",29.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""town""",71.0,"""7-9 grades of school [unfinish…","""In a registered marriage""","""MALE""","""Yes""","""You are not working""",19000.0,null,18000.0,"""Yes""","""No""","""No""","""N

In [153]:
panel['h7_2'].unique()

h7_2
str
"""November"""
"""September"""
"""December"""


In [154]:
panel = panel.with_columns(
    is_employed = pl.when(pl.col('j1').is_in(['You are currently working', 'You are on paid leave: maternity leave or taking care of a child under 3 years o', 'You are on another kind of paid leave'])).then(1).otherwise(0)
)
panel

idind,year,h7_2,h7_1,origsm,region,psu,status,age,educ,marst,h5,m20_7,j1,j60,j40,j363,m20_61,m20_62,m20_63,m20_64,m20_65,m20_66,m20_612,m20_613,m20_615,m20_8,chronic_heart,chronic_lung,chronic_liver,chronic_kidney,chronic_stomach,chronic_spinal,disease_neuro,disease_eye,allegies,disability_class,is_employed
f64,f64,str,f64,str,str,str,str,f64,str,str,str,str,str,f64,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i32
9.0,2019.0,"""November""",13.0,"""Yes, it is an adress from the …","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",73.0,"""Technical, medical, musis etc …","""In a registered marriage""","""FEMALE""","""No""","""You are not working""",16000.0,null,16000.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""","""no""",0
28.0,2019.0,"""November""",3.0,"""No, it is not belong to the re…","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",51.5,"""7-9 grades of school [unfinish…","""In a registered marriage""","""MALE""","""No""","""You are not working""",null,null,null,"""No""","""No""","""Yes""","""No""","""No""","""Yes""","""No""","""No""","""Yes""",null,"""No""","""No""","""Yes""","""No""","""No""","""Yes""","""No""","""No""","""Yes""","""no""",0
30.0,2019.0,"""November""",3.0,"""No, it is not belong to the re…","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",25.0,"""10 and more grades of school w…","""Never married""","""MALE""","""No""","""You are currently working""",null,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",1
36.0,2019.0,"""November""",1.0,"""Yes, it is an adress from the …","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",65.0,"""8 grades of school""","""In a registered marriage""","""FEMALE""","""No""","""You are not working""",14700.0,null,14700.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""no""",0
24104.0,2019.0,"""November""",1.0,"""No, it is not belong to the re…","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",55.5,"""Technical, medical, musis etc …","""Divorsed and not remarried""","""FEMALE""","""No""","""You are currently working""",20200.0,null,8200.0,"""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""Yes""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""Yes""","""No""","""no""",1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
59408.0,2020.0,"""September""",12.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""PGT""",69.0,"""Secondary School Diploma""","""Widower or widow""","""FEMALE""","""No""","""You are not working""",11660.0,null,10660.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",0
59409.0,2020.0,"""September""",12.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""PGT""",49.0,"""10 and more grades of school &…","""Divorsed and not remarried""","""MALE""","""No""","""You are not working""",0.0,null,null,"""No""","""No""","""No""","""No""","""No""",null,"""No""",null,"""No""",null,"""No""","""No""","""No""","""No""","""No""","""no""","""No""","""no""","""No""","""no""",0
59411.0,2020.0,"""November""",29.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""town""",71.0,"""7-9 grades of school [unfinish…","""In a registered marriage""","""MALE""","""Yes""","""You are not working""",19000.0,null,18000.0,

In [155]:
panel = panel.with_columns(
    has_disablity = pl.when(pl.col('m20_7').is_in(['No', 'NO ANSWER']) | pl.col('m20_7').is_null()).then(0).otherwise(1)
)
# panel = panel.with_columns(
#     pl.when(pl.col(''))
# )
panel

idind,year,h7_2,h7_1,origsm,region,psu,status,age,educ,marst,h5,m20_7,j1,j60,j40,j363,m20_61,m20_62,m20_63,m20_64,m20_65,m20_66,m20_612,m20_613,m20_615,m20_8,chronic_heart,chronic_lung,chronic_liver,chronic_kidney,chronic_stomach,chronic_spinal,disease_neuro,disease_eye,allegies,disability_class,is_employed,has_disablity
f64,f64,str,f64,str,str,str,str,f64,str,str,str,str,str,f64,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i32,i32
9.0,2019.0,"""November""",13.0,"""Yes, it is an adress from the …","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",73.0,"""Technical, medical, musis etc …","""In a registered marriage""","""FEMALE""","""No""","""You are not working""",16000.0,null,16000.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""","""no""",0,0
28.0,2019.0,"""November""",3.0,"""No, it is not belong to the re…","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",51.5,"""7-9 grades of school [unfinish…","""In a registered marriage""","""MALE""","""No""","""You are not working""",null,null,null,"""No""","""No""","""Yes""","""No""","""No""","""Yes""","""No""","""No""","""Yes""",null,"""No""","""No""","""Yes""","""No""","""No""","""Yes""","""No""","""No""","""Yes""","""no""",0,0
30.0,2019.0,"""November""",3.0,"""No, it is not belong to the re…","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",25.0,"""10 and more grades of school w…","""Never married""","""MALE""","""No""","""You are currently working""",null,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",1,0
36.0,2019.0,"""November""",1.0,"""Yes, it is an adress from the …","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",65.0,"""8 grades of school""","""In a registered marriage""","""FEMALE""","""No""","""You are not working""",14700.0,null,14700.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""no""",0,0
24104.0,2019.0,"""November""",1.0,"""No, it is not belong to the re…","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",55.5,"""Technical, medical, musis etc …","""Divorsed and not remarried""","""FEMALE""","""No""","""You are currently working""",20200.0,null,8200.0,"""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""Yes""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""Yes""","""No""","""no""",1,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
59408.0,2020.0,"""September""",12.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""PGT""",69.0,"""Secondary School Diploma""","""Widower or widow""","""FEMALE""","""No""","""You are not working""",11660.0,null,10660.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",0,0
59409.0,2020.0,"""September""",12.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""PGT""",49.0,"""10 and more grades of school &…","""Divorsed and not remarried""","""MALE""","""No""","""You are not working""",0.0,null,null,"""No""","""No""","""No""","""No""","""No""",null,"""No""",null,"""No""",null,"""No""","""No""","""No""","""No""","""No""","""no""","""No""","""no""","""No""","""no""",0,0
59411.0,2020.0,"""November""",29.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""town""",71.0,"""7-9 grades of school [unfinish…","""In a registered marriage""","""MALE""","""Yes""","""You are no

In [156]:
panel = panel.with_columns(
    is_employed_has_disablity = pl.col('is_employed') * pl.col('has_disablity')
)
panel

idind,year,h7_2,h7_1,origsm,region,psu,status,age,educ,marst,h5,m20_7,j1,j60,j40,j363,m20_61,m20_62,m20_63,m20_64,m20_65,m20_66,m20_612,m20_613,m20_615,m20_8,chronic_heart,chronic_lung,chronic_liver,chronic_kidney,chronic_stomach,chronic_spinal,disease_neuro,disease_eye,allegies,disability_class,is_employed,has_disablity,is_employed_has_disablity
f64,f64,str,f64,str,str,str,str,f64,str,str,str,str,str,f64,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i32,i32,i32
9.0,2019.0,"""November""",13.0,"""Yes, it is an adress from the …","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",73.0,"""Technical, medical, musis etc …","""In a registered marriage""","""FEMALE""","""No""","""You are not working""",16000.0,null,16000.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""","""no""",0,0,0
28.0,2019.0,"""November""",3.0,"""No, it is not belong to the re…","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",51.5,"""7-9 grades of school [unfinish…","""In a registered marriage""","""MALE""","""No""","""You are not working""",null,null,null,"""No""","""No""","""Yes""","""No""","""No""","""Yes""","""No""","""No""","""Yes""",null,"""No""","""No""","""Yes""","""No""","""No""","""Yes""","""No""","""No""","""Yes""","""no""",0,0,0
30.0,2019.0,"""November""",3.0,"""No, it is not belong to the re…","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",25.0,"""10 and more grades of school w…","""Never married""","""MALE""","""No""","""You are currently working""",null,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",1,0,0
36.0,2019.0,"""November""",1.0,"""Yes, it is an adress from the …","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",65.0,"""8 grades of school""","""In a registered marriage""","""FEMALE""","""No""","""You are not working""",14700.0,null,14700.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""no""",0,0,0
24104.0,2019.0,"""November""",1.0,"""No, it is not belong to the re…","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",55.5,"""Technical, medical, musis etc …","""Divorsed and not remarried""","""FEMALE""","""No""","""You are currently working""",20200.0,null,8200.0,"""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""Yes""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""Yes""","""No""","""no""",1,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
59408.0,2020.0,"""September""",12.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""PGT""",69.0,"""Secondary School Diploma""","""Widower or widow""","""FEMALE""","""No""","""You are not working""",11660.0,null,10660.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",0,0,0
59409.0,2020.0,"""September""",12.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""PGT""",49.0,"""10 and more grades of school &…","""Divorsed and not remarried""","""MALE""","""No""","""You are not working""",0.0,null,null,"""No""","""No""","""No""","""No""","""No""",null,"""No""",null,"""No""",null,"""No""","""No""","""No""","""No""","""No""","""no""","""No""","""no""","""No""","""no""",0,0,0
59411.0,2020.0,"""November""",29.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""town""",71.0,"""7-9 grades of school [unfinish…","""In a registered 

In [157]:
panel = panel.with_columns(
    is_employed_no_disablity = pl.col('is_employed') * pl.col('has_disablity').replace({0:1, 1:0})
)
panel

idind,year,h7_2,h7_1,origsm,region,psu,status,age,educ,marst,h5,m20_7,j1,j60,j40,j363,m20_61,m20_62,m20_63,m20_64,m20_65,m20_66,m20_612,m20_613,m20_615,m20_8,chronic_heart,chronic_lung,chronic_liver,chronic_kidney,chronic_stomach,chronic_spinal,disease_neuro,disease_eye,allegies,disability_class,is_employed,has_disablity,is_employed_has_disablity,is_employed_no_disablity
f64,f64,str,f64,str,str,str,str,f64,str,str,str,str,str,f64,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i32,i32,i32,i32
9.0,2019.0,"""November""",13.0,"""Yes, it is an adress from the …","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",73.0,"""Technical, medical, musis etc …","""In a registered marriage""","""FEMALE""","""No""","""You are not working""",16000.0,null,16000.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""","""no""",0,0,0,0
28.0,2019.0,"""November""",3.0,"""No, it is not belong to the re…","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",51.5,"""7-9 grades of school [unfinish…","""In a registered marriage""","""MALE""","""No""","""You are not working""",null,null,null,"""No""","""No""","""Yes""","""No""","""No""","""Yes""","""No""","""No""","""Yes""",null,"""No""","""No""","""Yes""","""No""","""No""","""Yes""","""No""","""No""","""Yes""","""no""",0,0,0,0
30.0,2019.0,"""November""",3.0,"""No, it is not belong to the re…","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",25.0,"""10 and more grades of school w…","""Never married""","""MALE""","""No""","""You are currently working""",null,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",1,0,0,1
36.0,2019.0,"""November""",1.0,"""Yes, it is an adress from the …","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",65.0,"""8 grades of school""","""In a registered marriage""","""FEMALE""","""No""","""You are not working""",14700.0,null,14700.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""no""",0,0,0,0
24104.0,2019.0,"""November""",1.0,"""No, it is not belong to the re…","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",55.5,"""Technical, medical, musis etc …","""Divorsed and not remarried""","""FEMALE""","""No""","""You are currently working""",20200.0,null,8200.0,"""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""Yes""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""Yes""","""No""","""no""",1,0,0,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
59408.0,2020.0,"""September""",12.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""PGT""",69.0,"""Secondary School Diploma""","""Widower or widow""","""FEMALE""","""No""","""You are not working""",11660.0,null,10660.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",0,0,0,0
59409.0,2020.0,"""September""",12.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""PGT""",49.0,"""10 and more grades of school &…","""Divorsed and not remarried""","""MALE""","""No""","""You are not working""",0.0,null,null,"""No""","""No""","""No""","""No""","""No""",null,"""No""",null,"""No""",null,"""No""","""No""","""No""","""No""","""No""","""no""","""No""","""no""","""No""","""no""",0,0,0,0
59411.0,2020.0,"""November""",29.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""town""",71.0,"""7-9 gra

In [158]:
panel = panel.with_columns(
    net_job = pl.col('j60').fill_null(0) - pl.col('j363').fill_null(0)
)
panel = panel.with_columns(
    pl.when(pl.col('net_job') < 0).then(0).otherwise('net_job').alias('net_job')
)
# panel = panel.with_columns(
#     pl.when(pl.col('is_employed') == 0).then(0).otherwise('net_job').alias('net_job')
# ) 
panel

idind,year,h7_2,h7_1,origsm,region,psu,status,age,educ,marst,h5,m20_7,j1,j60,j40,j363,m20_61,m20_62,m20_63,m20_64,m20_65,m20_66,m20_612,m20_613,m20_615,m20_8,chronic_heart,chronic_lung,chronic_liver,chronic_kidney,chronic_stomach,chronic_spinal,disease_neuro,disease_eye,allegies,disability_class,is_employed,has_disablity,is_employed_has_disablity,is_employed_no_disablity,net_job
f64,f64,str,f64,str,str,str,str,f64,str,str,str,str,str,f64,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i32,i32,i32,i32,f64
9.0,2019.0,"""November""",13.0,"""Yes, it is an adress from the …","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",73.0,"""Technical, medical, musis etc …","""In a registered marriage""","""FEMALE""","""No""","""You are not working""",16000.0,null,16000.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""","""no""",0,0,0,0,0.0
28.0,2019.0,"""November""",3.0,"""No, it is not belong to the re…","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",51.5,"""7-9 grades of school [unfinish…","""In a registered marriage""","""MALE""","""No""","""You are not working""",null,null,null,"""No""","""No""","""Yes""","""No""","""No""","""Yes""","""No""","""No""","""Yes""",null,"""No""","""No""","""Yes""","""No""","""No""","""Yes""","""No""","""No""","""Yes""","""no""",0,0,0,0,0.0
30.0,2019.0,"""November""",3.0,"""No, it is not belong to the re…","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",25.0,"""10 and more grades of school w…","""Never married""","""MALE""","""No""","""You are currently working""",null,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",1,0,0,1,0.0
36.0,2019.0,"""November""",1.0,"""Yes, it is an adress from the …","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",65.0,"""8 grades of school""","""In a registered marriage""","""FEMALE""","""No""","""You are not working""",14700.0,null,14700.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""no""",0,0,0,0,0.0
24104.0,2019.0,"""November""",1.0,"""No, it is not belong to the re…","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",55.5,"""Technical, medical, musis etc …","""Divorsed and not remarried""","""FEMALE""","""No""","""You are currently working""",20200.0,null,8200.0,"""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""Yes""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""Yes""","""No""","""no""",1,0,0,1,12000.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
59408.0,2020.0,"""September""",12.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""PGT""",69.0,"""Secondary School Diploma""","""Widower or widow""","""FEMALE""","""No""","""You are not working""",11660.0,null,10660.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",0,0,0,0,1000.0
59409.0,2020.0,"""September""",12.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""PGT""",49.0,"""10 and more grades of school &…","""Divorsed and not remarried""","""MALE""","""No""","""You are not working""",0.0,null,null,"""No""","""No""","""No""","""No""","""No""",null,"""No""",null,"""No""",null,"""No""","""No""","""No""","""No""","""No""","""no""","""No""","""no""","""No""","""no""",0,0,0,0,0.0
59411.0,2020.0,"""November""",29.0,"""Yes, it is an adress from the …","""Moscow Oblast""",""

In [159]:
panel  = panel.filter(pl.col('net_job') < 1_000_000)
panel

idind,year,h7_2,h7_1,origsm,region,psu,status,age,educ,marst,h5,m20_7,j1,j60,j40,j363,m20_61,m20_62,m20_63,m20_64,m20_65,m20_66,m20_612,m20_613,m20_615,m20_8,chronic_heart,chronic_lung,chronic_liver,chronic_kidney,chronic_stomach,chronic_spinal,disease_neuro,disease_eye,allegies,disability_class,is_employed,has_disablity,is_employed_has_disablity,is_employed_no_disablity,net_job
f64,f64,str,f64,str,str,str,str,f64,str,str,str,str,str,f64,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i32,i32,i32,i32,f64
9.0,2019.0,"""November""",13.0,"""Yes, it is an adress from the …","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",73.0,"""Technical, medical, musis etc …","""In a registered marriage""","""FEMALE""","""No""","""You are not working""",16000.0,null,16000.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""","""no""",0,0,0,0,0.0
28.0,2019.0,"""November""",3.0,"""No, it is not belong to the re…","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",51.5,"""7-9 grades of school [unfinish…","""In a registered marriage""","""MALE""","""No""","""You are not working""",null,null,null,"""No""","""No""","""Yes""","""No""","""No""","""Yes""","""No""","""No""","""Yes""",null,"""No""","""No""","""Yes""","""No""","""No""","""Yes""","""No""","""No""","""Yes""","""no""",0,0,0,0,0.0
30.0,2019.0,"""November""",3.0,"""No, it is not belong to the re…","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",25.0,"""10 and more grades of school w…","""Never married""","""MALE""","""No""","""You are currently working""",null,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",1,0,0,1,0.0
36.0,2019.0,"""November""",1.0,"""Yes, it is an adress from the …","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",65.0,"""8 grades of school""","""In a registered marriage""","""FEMALE""","""No""","""You are not working""",14700.0,null,14700.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""no""",0,0,0,0,0.0
24104.0,2019.0,"""November""",1.0,"""No, it is not belong to the re…","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",55.5,"""Technical, medical, musis etc …","""Divorsed and not remarried""","""FEMALE""","""No""","""You are currently working""",20200.0,null,8200.0,"""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""Yes""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""Yes""","""No""","""no""",1,0,0,1,12000.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
59408.0,2020.0,"""September""",12.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""PGT""",69.0,"""Secondary School Diploma""","""Widower or widow""","""FEMALE""","""No""","""You are not working""",11660.0,null,10660.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",0,0,0,0,1000.0
59409.0,2020.0,"""September""",12.0,"""Yes, it is an adress from the …","""Moscow Oblast""","""Moscovskaya Oblast""","""PGT""",49.0,"""10 and more grades of school &…","""Divorsed and not remarried""","""MALE""","""No""","""You are not working""",0.0,null,null,"""No""","""No""","""No""","""No""","""No""",null,"""No""",null,"""No""",null,"""No""","""No""","""No""","""No""","""No""","""no""","""No""","""no""","""No""","""no""",0,0,0,0,0.0
59411.0,2020.0,"""November""",29.0,"""Yes, it is an adress from the …","""Moscow Oblast""",""

In [160]:
panel.filter(pl.col('age') >= 18).filter(pl.col('has_disablity') == 1).sort('age')

idind,year,h7_2,h7_1,origsm,region,psu,status,age,educ,marst,h5,m20_7,j1,j60,j40,j363,m20_61,m20_62,m20_63,m20_64,m20_65,m20_66,m20_612,m20_613,m20_615,m20_8,chronic_heart,chronic_lung,chronic_liver,chronic_kidney,chronic_stomach,chronic_spinal,disease_neuro,disease_eye,allegies,disability_class,is_employed,has_disablity,is_employed_has_disablity,is_employed_no_disablity,net_job
f64,f64,str,f64,str,str,str,str,f64,str,str,str,str,str,f64,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i32,i32,i32,i32,f64
23081.0,2019.0,"""November""",20.0,"""No, it is not belong to the re…","""Altaiskij Kraj: Kur`inskij Raj…","""Kur`inskij Rajon: Altaiskij Kr…","""rural""",18.0,"""9 grades of school""","""Never married""","""FEMALE""","""Yes""","""You are not working""",18900.0,null,18900.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Second group""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Second group""",0,1,0,0,0.0
40121.0,2019.0,"""November""",11.0,"""Yes, it is an adress from the …","""Moscow City""","""Moscow City""","""oblastnoy center""",18.0,"""10 and more grades of school w…","""Never married""","""MALE""","""Yes""","""You are not working""",7000.0,null,null,"""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""Yes""","""Third group""","""No""","""No""","""no""","""No""","""No""","""No""","""No""","""No""","""Yes""","""Third group""",0,1,0,0,7000.0
46172.0,2020.0,"""November""",30.0,"""Yes, it is an adress from the …","""Cheliabinsk""","""Cheliabinsk: Chelyabinskaya Ob…","""oblastnoy center""",18.0,"""8 grades of school""","""Never married""","""MALE""","""Yes""","""You are not working""",9000.0,null,9000.0,"""Yes""",null,"""Yes""","""Yes""","""Yes""","""No""","""Yes""","""Yes""","""Yes""","""First group""","""Yes""","""no""","""Yes""","""Yes""","""Yes""","""No""","""Yes""","""Yes""","""Yes""","""First group""",0,1,0,0,0.0
37791.0,2020.0,"""September""",20.0,"""Yes, it is an adress from the …","""Cheliabinsk Oblast: Krasnoarm…","""Krasnoarmeyskiy Rajon: Chelyab…","""rural""",18.0,"""9 grades of school""","""Never married""","""FEMALE""","""Yes""","""You are not working""",7000.0,null,7000.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Second group""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Second group""",0,1,0,0,0.0
32522.0,2020.0,"""September""",22.0,"""Yes, it is an adress from the …","""Krasnodar CR""","""Krasnodar: Krasnodarskij Krai""","""oblastnoy center""",18.5,"""10 and more grades of school w…","""Never married""","""FEMALE""","""Yes""","""You are not working""",11000.0,null,11000.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",0,1,0,0,0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
27308.0,2020.0,"""November""",30.0,"""Yes, it is an adress from the …","""Cheliabinsk""","""Cheliabinsk: Chelyabinskaya Ob…","""oblastnoy center""",93.0,"""Institute, University, Academy…","""In a registered marriage""","""MALE""","""Yes""","""You are not working""",27940.0,null,26940.0,"""No""","""No""","""Yes""","""Yes""","""No""","""No""","""No""","""Yes""","""No""","""Third group""","""No""","""No""","""Yes""","""Yes""","""No""","""No""","""No""","""Yes""","""No""","""Third group""",0,1,0,0,1000.0
5444.0,2020.0,"""September""",25.0,"""Yes, it is an adress from the …","""Altaiskij Kraj: Biisk CR""","""Biisk City and Rajon: Altaiski…","""town""",93.5,"""7 grades of school""","""Never married""","""FEMALE""","""Yes""","""You are not working""",24200.0,null,23600.0,"""Yes""","""No""","""No""","""No""","""No""","""Yes""","""Yes""","""Yes""","""No""","""Second group""","""Yes""","""No""","""No""","""No""","""No""","""Yes""","""Yes""","""Yes""","""No""","""Second 

In [161]:
panel.filter(pl.col('age') >= 18).filter(pl.col('has_disablity') == 1).sort('age')['disability_class'].value_counts()

disability_class,count
str,u32
"""Third group""",528
"""First group""",104
"""no""",10
"""Second group""",713


In [162]:
len(panel['idind'].unique())

10912

In [163]:
common_people = panel.filter(pl.col('year') == 2019)[['idind']].join(panel.filter(pl.col('year') == 2020)[['idind']], how='inner', on='idind').sort('idind').unique()
common_people

idind
f64
36.0
60.0
125.0
126.0
127.0
…
59274.0
59275.0
59277.0


In [167]:
panel_common_2019_2020 = panel.filter(
    pl.col('idind').is_in(common_people['idind'].implode())
).sort('idind')
panel_common_2019_2020

idind,year,h7_2,h7_1,origsm,region,psu,status,age,educ,marst,h5,m20_7,j1,j60,j40,j363,m20_61,m20_62,m20_63,m20_64,m20_65,m20_66,m20_612,m20_613,m20_615,m20_8,chronic_heart,chronic_lung,chronic_liver,chronic_kidney,chronic_stomach,chronic_spinal,disease_neuro,disease_eye,allegies,disability_class,is_employed,has_disablity,is_employed_has_disablity,is_employed_no_disablity,net_job
f64,f64,str,f64,str,str,str,str,f64,str,str,str,str,str,f64,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i32,i32,i32,i32,f64
36.0,2019.0,"""November""",1.0,"""Yes, it is an adress from the …","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",65.0,"""8 grades of school""","""In a registered marriage""","""FEMALE""","""No""","""You are not working""",14700.0,null,14700.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""no""",0,0,0,0,0.0
36.0,2020.0,"""September""",29.0,"""Yes, it is an adress from the …","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",66.0,"""8 grades of school""","""In a registered marriage""","""FEMALE""","""No""","""You are not working""",16300.0,null,16300.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""no""",0,0,0,0,0.0
60.0,2019.0,"""November""",4.0,"""Yes, it is an adress from the …","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",51.0,"""10 and more grades of school &…","""In a registered marriage""","""FEMALE""","""No""","""You are currently working""",18000.0,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",1,0,0,1,18000.0
60.0,2020.0,"""September""",30.0,"""Yes, it is an adress from the …","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",52.0,"""10 and more grades of school &…","""In a registered marriage""","""FEMALE""","""No""","""You are currently working""",30000.0,null,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,null,"""no""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""","""no""",1,0,0,1,30000.0
125.0,2019.0,"""November""",13.0,"""Yes, it is an adress from the …","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""rural""",56.0,"""Secondary School Diploma""","""In a registered marriage""","""FEMALE""","""No""","""You are currently working""",35000.0,null,15000.0,"""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""No""","""No""","""no""",1,0,0,1,20000.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
59277.0,2020.0,"""September""",12.0,"""Yes, it is an adress from the …","""Moscow City""","""Moscow City""","""oblastnoy center""",33.5,"""Institute, University, Academy…","""Never married""","""FEMALE""","""No""","""You are currently working""",40000.0,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""no""",1,0,0,1,40000.0
59278.0,2019.0,"""November""",4.0,"""Yes, it is an adress from the …","""Moscow City""","""Moscow City""","""oblastnoy center""",19.5,"""Technical, medical, musis etc …","""Never married""","""MALE""","""Yes""","""You are not working""",22000.0,null,22000.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Third group""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Third group""",0,1,0,0,0.0
59278.0,2020.0,"""September""",30.0,"""No, it is 

In [168]:
panel_common_2019_2020 = (
    panel_common_2019_2020
        .join(education_levels, on='educ', how='left', validate='m:1')
        .with_columns(pl.col('educ_level').fill_null('school_or_less'))
        .join(marital_status, on='marst', how='left', validate='m:1')
        .join(period_relevance, on='year', how='left', validate='m:1')
        .join(is_town, on='status', how='left', validate='m:1')
        .with_columns(pl.col('is_married').fill_null(0))
        .with_columns(
            is_female = pl.when(pl.col('h5') == 'FEMALE').then(1).otherwise(0),
        ).with_columns(
            pl.when(pl.col('chronic_heart') == 'Yes').then(1).otherwise(0).alias('chronic_heart'),
            pl.when(pl.col('chronic_lung') == 'Yes').then(1).otherwise(0).alias('chronic_lung'),
            pl.when(pl.col('chronic_liver') == 'Yes').then(1).otherwise(0).alias('chronic_liver'),
            pl.when(pl.col('chronic_kidney') == 'Yes').then(1).otherwise(0).alias('chronic_kidney'),
            pl.when(pl.col('chronic_stomach') == 'Yes').then(1).otherwise(0).alias('chronic_stomach'),
            pl.when(pl.col('chronic_spinal') == 'Yes').then(1).otherwise(0).alias('chronic_spinal'),
            pl.when(pl.col('disease_neuro') == 'Yes').then(1).otherwise(0).alias('disease_neuro'),
            pl.when(pl.col('disease_eye') == 'Yes').then(1).otherwise(0).alias('disease_eye'),
            pl.when(pl.col('allegies') == 'Yes').then(1).otherwise(0).alias('allegies')
        )
        #.filter(pl.col('idind').is_in(common_people))
)#.drop(cols_disease) 
panel_common_2019_2020

idind,year,h7_2,h7_1,origsm,region,psu,status,age,educ,marst,h5,m20_7,j1,j60,j40,j363,m20_61,m20_62,m20_63,m20_64,m20_65,m20_66,m20_612,m20_613,m20_615,m20_8,chronic_heart,chronic_lung,chronic_liver,chronic_kidney,chronic_stomach,chronic_spinal,disease_neuro,disease_eye,allegies,disability_class,is_employed,has_disablity,is_employed_has_disablity,is_employed_no_disablity,net_job,educ_level,is_married,period_relevance,is_town,is_female
f64,f64,str,f64,str,str,str,str,f64,str,str,str,str,str,f64,f64,f64,str,str,str,str,str,str,str,str,str,str,i32,i32,i32,i32,i32,i32,i32,i32,i32,str,i32,i32,i32,i32,f64,str,i64,i64,i64,i32
36.0,2019.0,"""November""",1.0,"""Yes, it is an adress from the …","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",65.0,"""8 grades of school""","""In a registered marriage""","""FEMALE""","""No""","""You are not working""",14700.0,null,14700.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""","""No""",null,0,0,0,0,0,0,0,1,0,"""no""",0,0,0,0,0.0,"""school_or_less""",1,0,1,1
36.0,2020.0,"""September""",29.0,"""Yes, it is an adress from the …","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",66.0,"""8 grades of school""","""In a registered marriage""","""FEMALE""","""No""","""You are not working""",16300.0,null,16300.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""","""No""",null,0,0,0,0,0,0,0,1,0,"""no""",0,0,0,0,0.0,"""school_or_less""",1,1,1,1
60.0,2019.0,"""November""",4.0,"""Yes, it is an adress from the …","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",51.0,"""10 and more grades of school &…","""In a registered marriage""","""FEMALE""","""No""","""You are currently working""",18000.0,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,0,0,0,0,0,0,0,0,0,"""no""",1,0,0,1,18000.0,"""higher""",1,0,1,1
60.0,2020.0,"""September""",30.0,"""Yes, it is an adress from the …","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",52.0,"""10 and more grades of school &…","""In a registered marriage""","""FEMALE""","""No""","""You are currently working""",30000.0,null,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,null,0,0,0,0,0,0,0,0,0,"""no""",1,0,0,1,30000.0,"""higher""",1,1,1,1
125.0,2019.0,"""November""",13.0,"""Yes, it is an adress from the …","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""rural""",56.0,"""Secondary School Diploma""","""In a registered marriage""","""FEMALE""","""No""","""You are currently working""",35000.0,null,15000.0,"""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""No""","""No""",null,0,0,0,0,0,1,0,0,0,"""no""",1,0,0,1,20000.0,"""common""",1,0,0,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
59277.0,2020.0,"""September""",12.0,"""Yes, it is an adress from the …","""Moscow City""","""Moscow City""","""oblastnoy center""",33.5,"""Institute, University, Academy…","""Never married""","""FEMALE""","""No""","""You are currently working""",40000.0,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,0,0,0,0,0,0,0,0,0,"""no""",1,0,0,1,40000.0,"""university""",0,1,1,1
59278.0,2019.0,"""November""",4.0,"""Yes, it is an adress from the …","""Moscow City""","""Moscow City""","""oblastnoy center""",19.5,"""Technical, medical, musis etc …","""Never married""","""MALE""","""Yes""","""You are not working""",22000.0,null,22000.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Third group""",0,0,0,0,0,0,0,0,0,"""Third group""",0,1,0,0,0.0,"""common""",0,0,1,0
59278.0,2020.0,"""September""",30.0,"""No, it is not belong to the re…","""Moscow City""","""Moscow City""","""oblastnoy center""",20.5,"""7-9 grades of school [unfinish…","""Never married""","""MALE""","""No""","""You are not working""",3

In [189]:
(
    panel_common_2019_2020[['year', 'has_disablity']]
    .group_by('year').agg(pl.col('has_disablity').sum())
)

year,has_disablity
f64,i32
2020.0,347
2019.0,338


In [169]:
working_statuses = (
    panel_common_2019_2020[['idind', 'year', 'is_employed']]
    .pivot(index='idind', on='year', values='is_employed')
                   )
working_statuses#['idind'].value_counts().sort('count')

idind,2019.0,2020.0
f64,i32,i32
36.0,0,0
60.0,1,1
125.0,1,1
126.0,1,1
127.0,0,1
…,…,…
59274.0,0,1
59275.0,1,1
59277.0,1,1


In [170]:
working_statuses_1 = working_statuses.filter(
    (pl.col('2019.0') == 1) | (pl.col('2020.0') == 1)
)
working_statuses_1

idind,2019.0,2020.0
f64,i32,i32
60.0,1,1
125.0,1,1
126.0,1,1
127.0,0,1
128.0,1,1
…,…,…
59271.0,1,1
59274.0,0,1
59275.0,1,1


In [191]:
disability_statuses = (
    panel_common_2019_2020[['idind', 'year', 'has_disablity']]
    .pivot(index='idind', on='year', values='has_disablity')
                   )
disability_statuses#['idind'].value_counts().sort('count')

idind,2019.0,2020.0
f64,i32,i32
36.0,0,0
60.0,0,0
125.0,0,0
126.0,0,0
127.0,0,0
…,…,…
59274.0,0,0
59275.0,0,0
59277.0,0,0


In [217]:
disability_statuses_common = disability_statuses.filter(
    (pl.col('2019.0') == 1) | (pl.col('2020.0') == 1)
)
disability_statuses_common

idind,2019.0,2020.0
f64,i32,i32
132.0,1,1
423.0,1,1
425.0,1,1
528.0,1,1
566.0,1,1
…,…,…
59039.0,0,1
59073.0,1,1
59145.0,1,1


In [218]:
panel_common_2019_2020 = panel_common_2019_2020.with_columns(
    pl.when(pl.col('idind').is_in(disability_statuses_common['idind'].implode()))
    .then(1).otherwise(0).alias('has_disability_common')
)
panel_common_2019_2020

idind,year,h7_2,h7_1,origsm,region,psu,status,age,educ,marst,h5,m20_7,j1,j60,j40,j363,m20_61,m20_62,m20_63,m20_64,m20_65,m20_66,m20_612,m20_613,m20_615,m20_8,chronic_heart,chronic_lung,chronic_liver,chronic_kidney,chronic_stomach,chronic_spinal,disease_neuro,disease_eye,allegies,disability_class,is_employed,has_disablity,is_employed_has_disablity,is_employed_no_disablity,net_job,educ_level,is_married,period_relevance,is_town,is_female,has_disability_2019,has_disability_common
f64,f64,str,f64,str,str,str,str,f64,str,str,str,str,str,f64,f64,f64,str,str,str,str,str,str,str,str,str,str,i32,i32,i32,i32,i32,i32,i32,i32,i32,str,i32,i32,i32,i32,f64,str,i64,i64,i64,i32,i32,i32
36.0,2019.0,"""November""",1.0,"""Yes, it is an adress from the …","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",65.0,"""8 grades of school""","""In a registered marriage""","""FEMALE""","""No""","""You are not working""",14700.0,null,14700.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""","""No""",null,0,0,0,0,0,0,0,1,0,"""no""",0,0,0,0,0.0,"""school_or_less""",1,0,1,1,0,0
36.0,2020.0,"""September""",29.0,"""Yes, it is an adress from the …","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",66.0,"""8 grades of school""","""In a registered marriage""","""FEMALE""","""No""","""You are not working""",16300.0,null,16300.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Yes""","""No""",null,0,0,0,0,0,0,0,1,0,"""no""",0,0,0,0,0.0,"""school_or_less""",1,1,1,1,0,0
60.0,2019.0,"""November""",4.0,"""Yes, it is an adress from the …","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",51.0,"""10 and more grades of school &…","""In a registered marriage""","""FEMALE""","""No""","""You are currently working""",18000.0,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,0,0,0,0,0,0,0,0,0,"""no""",1,0,0,1,18000.0,"""higher""",1,0,1,1,0,0
60.0,2020.0,"""September""",30.0,"""Yes, it is an adress from the …","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",52.0,"""10 and more grades of school &…","""In a registered marriage""","""FEMALE""","""No""","""You are currently working""",30000.0,null,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,null,0,0,0,0,0,0,0,0,0,"""no""",1,0,0,1,30000.0,"""higher""",1,1,1,1,0,0
125.0,2019.0,"""November""",13.0,"""Yes, it is an adress from the …","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""rural""",56.0,"""Secondary School Diploma""","""In a registered marriage""","""FEMALE""","""No""","""You are currently working""",35000.0,null,15000.0,"""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""No""","""No""",null,0,0,0,0,0,1,0,0,0,"""no""",1,0,0,1,20000.0,"""common""",1,0,0,1,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
59277.0,2020.0,"""September""",12.0,"""Yes, it is an adress from the …","""Moscow City""","""Moscow City""","""oblastnoy center""",33.5,"""Institute, University, Academy…","""Never married""","""FEMALE""","""No""","""You are currently working""",40000.0,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,0,0,0,0,0,0,0,0,0,"""no""",1,0,0,1,40000.0,"""university""",0,1,1,1,0,0
59278.0,2019.0,"""November""",4.0,"""Yes, it is an adress from the …","""Moscow City""","""Moscow City""","""oblastnoy center""",19.5,"""Technical, medical, musis etc …","""Never married""","""MALE""","""Yes""","""You are not working""",22000.0,null,22000.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""Third group""",0,0,0,0,0,0,0,0,0,"""Third group""",0,1,0,0,0.0,"""common""",0,0,1,0,1,1
59278.0,2020.0,"""September""",30.0,"""No, it is not belong to the re…","""Moscow City""","""Moscow City""","""oblastnoy center""",20.5,"""7-9 grades of scho

In [219]:
(
    panel_common_2019_2020
        #.filter((pl.col('is_employed_no_disablity') == 1)|(pl.col('is_employed_has_disablity') == 1))
        #.filter(pl.col('h7_2').is_in(['October', 'November', 'December']))
        .filter(pl.col('idind').is_in(working_statuses_1['idind'].implode()))
        .with_columns(pl.col('year').cast(pl.Int32).cast(pl.String).str.to_date("%Y"))
        .group_by(['year', 'has_disablity']).agg(pl.col('net_job').mean())
).plot.line(x='year', y='net_job', color='has_disablity')

alt.Chart(...)

In [220]:
(
    panel_common_2019_2020
        #.filter((pl.col('is_employed_no_disablity') == 1)|(pl.col('is_employed_has_disablity') == 1))
        #.filter(pl.col('h7_2').is_in(['October', 'November', 'December']))
        .filter(pl.col('idind').is_in(working_statuses_1['idind'].implode()))
        .with_columns(pl.col('year').cast(pl.Int32).cast(pl.String).str.to_date("%Y"))
        .group_by(['year', 'has_disability_common']).agg(pl.col('net_job').mean())
).plot.line(x='year', y='net_job', color='has_disability_common')

alt.Chart(...)

In [221]:
working_data = (
    panel_common_2019_2020
    .filter(pl.col('idind').is_in(working_statuses_1['idind'].implode()))
    .sort(['idind', 'year'])
)
working_data

idind,year,h7_2,h7_1,origsm,region,psu,status,age,educ,marst,h5,m20_7,j1,j60,j40,j363,m20_61,m20_62,m20_63,m20_64,m20_65,m20_66,m20_612,m20_613,m20_615,m20_8,chronic_heart,chronic_lung,chronic_liver,chronic_kidney,chronic_stomach,chronic_spinal,disease_neuro,disease_eye,allegies,disability_class,is_employed,has_disablity,is_employed_has_disablity,is_employed_no_disablity,net_job,educ_level,is_married,period_relevance,is_town,is_female,has_disability_2019,has_disability_common
f64,f64,str,f64,str,str,str,str,f64,str,str,str,str,str,f64,f64,f64,str,str,str,str,str,str,str,str,str,str,i32,i32,i32,i32,i32,i32,i32,i32,i32,str,i32,i32,i32,i32,f64,str,i64,i64,i64,i32,i32,i32
60.0,2019.0,"""November""",4.0,"""Yes, it is an adress from the …","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",51.0,"""10 and more grades of school &…","""In a registered marriage""","""FEMALE""","""No""","""You are currently working""",18000.0,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,0,0,0,0,0,0,0,0,0,"""no""",1,0,0,1,18000.0,"""higher""",1,0,1,1,0,0
60.0,2020.0,"""September""",30.0,"""Yes, it is an adress from the …","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""PGT""",52.0,"""10 and more grades of school &…","""In a registered marriage""","""FEMALE""","""No""","""You are currently working""",30000.0,null,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,null,0,0,0,0,0,0,0,0,0,"""no""",1,0,0,1,30000.0,"""higher""",1,1,1,1,0,0
125.0,2019.0,"""November""",13.0,"""Yes, it is an adress from the …","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""rural""",56.0,"""Secondary School Diploma""","""In a registered marriage""","""FEMALE""","""No""","""You are currently working""",35000.0,null,15000.0,"""No""","""No""","""No""","""No""","""No""","""Yes""","""No""","""No""","""No""",null,0,0,0,0,0,1,0,0,0,"""no""",1,0,0,1,20000.0,"""common""",1,0,0,1,0,0
125.0,2020.0,"""November""",9.0,"""Yes, it is an adress from the …","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""rural""",57.0,"""Secondary School Diploma""","""In a registered marriage""","""FEMALE""","""No""","""You are currently working""",45500.0,null,16000.0,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,0,0,0,0,0,0,0,0,0,"""no""",1,0,0,1,29500.0,"""common""",1,1,0,1,0,0
126.0,2019.0,"""November""",13.0,"""Yes, it is an adress from the …","""Leningrad Oblast: Volosovkij R…","""Volosovkij Rajon: Leningradska…","""rural""",50.5,"""10 and more grades of school &…","""In a registered marriage""","""MALE""","""No""","""You are currently working""",20000.0,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,0,0,0,0,0,0,0,0,0,"""no""",1,0,0,1,20000.0,"""higher""",1,0,0,0,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
59275.0,2020.0,"""September""",29.0,"""Yes, it is an adress from the …","""Novosibirskaya Oblast: Berdsk …","""Tumenskaya Obl, Surgutskij R-n…","""town""",25.5,"""Secondary School Diploma""","""Divorsed and not remarried""","""MALE""","""No""","""You are currently working""",27000.0,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,0,0,0,0,0,0,0,0,0,"""no""",1,0,0,1,27000.0,"""common""",0,1,1,0,0,0
59277.0,2019.0,"""November""",21.0,"""Yes, it is an adress from the …","""Moscow City""","""Moscow City""","""oblastnoy center""",32.5,"""Institute, University, Academy…","""Never married""","""FEMALE""","""No""","""You are currently working""",40000.0,null,null,"""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""","""No""",null,0,0,0,0,0,0,0,0,0,"""no""",1,0,0,1,40000.0,"""university""",0,0,1,1,0,0
59277.0,2020.0,"""September""",12.0,"""Yes, it is an adress from the …","""Moscow City""","""Moscow City""","""oblastnoy 

In [222]:
working_data.filter(pl.col('has_disablity') == 1).plot.boxplot(y='net_job').properties(width=100, height=350)

alt.Chart(...)

In [223]:
working_data.filter(pl.col('has_disablity') == 0).plot.boxplot(y='net_job').properties(width=100, height=350)

alt.Chart(...)

In [224]:
ids_to_exclude_0 = working_data.filter(pl.col('has_disablity') == 0).filter(pl.col('net_job') > 500_000)['idind'].to_list()
#ids_to_exclude_0 = []
ids_to_exclude_0

[51497.0]

In [225]:
ids_to_exclude_1 = working_data.filter(pl.col('has_disablity') == 1).filter(pl.col('net_job') > 100_000)['idind'].to_list()
ids_to_exclude_1 = [x for x in ids_to_exclude_1 if x not in [7399, 40749]]
ids_to_exclude_1

[]

In [226]:
(
    working_data
        .filter(~pl.col('idind').is_in(ids_to_exclude_0 + ids_to_exclude_1))
        .with_columns(pl.col('year').cast(pl.Int32).cast(pl.String).str.to_date("%Y"))
        .group_by(['year', 'has_disablity']).agg(pl.col('net_job').mean())
).plot.line(x='year', y='net_job', color='has_disablity')

alt.Chart(...)

In [227]:
(
    working_data
        .filter(~pl.col('idind').is_in(ids_to_exclude_0 + ids_to_exclude_1))
        .with_columns(pl.col('year').cast(pl.Int32).cast(pl.String).str.to_date("%Y"))
        .group_by(['year', 'has_disability_common']).agg(pl.col('net_job').mean())
).plot.line(x='year', y='net_job', color='has_disability_common')

alt.Chart(...)

In [228]:
panel2019_2020 = (
    working_data
        # .filter(pl.col())
        .filter(~pl.col('idind').is_in(ids_to_exclude_0 + ids_to_exclude_1))
        .with_columns(pl.col('year').cast(pl.Int32).cast(pl.String).str.to_date("%Y"))
).to_pandas().set_index(['idind', 'year'])
panel2019_2020

h7_2  h7_1  \
idind   year                          
60.0    2019-01-01   November   4.0   
        2020-01-01  September  30.0   
125.0   2019-01-01   November  13.0   
        2020-01-01   November   9.0   
126.0   2019-01-01   November  13.0   
...                       ...   ...   
59275.0 2020-01-01  September  29.0   
59277.0 2019-01-01   November  21.0   
        2020-01-01  September  12.0   
59285.0 2019-01-01   November  21.0   
        2020-01-01  September  27.0   

                                                               origsm  \
idind   year                                                            
60.0    2019-01-01  Yes, it is an adress from the representative s...   
        2020-01-01  Yes, it is an adress from the representative s...   
125.0   2019-01-01  Yes, it is an adress from the representative s...   
        2020-01-01  Yes, it is an adress from the representative s...   
126.0   2019-01-01  Yes, it is an adress from the representative s...   
...                                                               ...   
59275.0 2020-01-01  Yes, it is an adress from the representative s...   
59277.0 2019-01-01  Yes, it is an adress from the representative s...   
        2020-01-01  Yes, it is an adress from the representative s...   
59285.0 2019-01-01  Yes, it is an adress from the representative s...   
        2020-01-01  Yes, it is an adress from the representative s...   

                                                               region  \
idind   year                                                            
60.0    2019-01-01                 Leningrad Oblast: Volosovkij Rajon   
        2020-01-01                 Leningrad Oblast: Volosovkij Rajon   
125.0   2019-01-01                 Leningrad Oblast: Volosovkij Rajon   
        2020-01-01                 Leningrad Oblast: Volosovkij Rajon   
126.0   2019-01-01                 Leningrad Oblast: Volosovkij Rajon   
...                                                               ...   
59275.0 2020-01-01  Novosibirskaya Oblast: Berdsk City & Raion (20...   
59277.0 2019-01-01                                        Moscow City   
        2020-01-01                                        Moscow City   
59285.0 2019-01-01                                        Moscow City   
        2020-01-01                                        Moscow City   

                                                                  psu  \
idind   year                                                            
60.0    2019-01-01            Volosovkij Rajon: Leningradskaya Oblast   
        2020-01-01            Volosovkij Rajon: Leningradskaya Oblast   
125.0   2019-01-01            Volosovkij Rajon: Leningradskaya Oblast   
        2020-01-01            Volosovkij Rajon: Leningradskaya Oblast   
126.0   2019-01-01            Volosovkij Rajon: Leningradskaya Oblast   
...                                                               ...   
59275.0 2020-01-01  Tumenskaya Obl, Surgutskij R-n: Khanty-Mansi A...   
59277.0 2019-01-01                                        Moscow City   
        2020-01-01                                        Moscow City   
59285.0 2019-01-01                                        Moscow City   
        2020-01-01                                        Moscow City   

                              status   age  \
idind   year                                 
60.0    2019-01-01               PGT  51.0   
        2020-01-01               PGT  52.0   
125.0   2019-01-01             rural  56.0   
        2020-01-01             rural  57.0   
126.0   2019-01-01             rural  50.5   
...                              ...   ...   
59275.0 2020-01-01              town  25.5   
59277.0 2019-01-01  oblastnoy center  32.5   
        2020-01-01  oblastnoy center  33.5   
59285.0 2019-01-01  oblastnoy center  41.0   
        2020-01-01  oblastnoy center  42.0   

                                                                 edu

In [239]:
# (
#     working_data
#         .filter(~pl.col('idind').is_in(ids_to_exclude_0 + ids_to_exclude_1))
# ).write_parquet('../data/working_data_with_disability_common.parquet')

In [229]:
# 
model1 = PanelOLS.from_formula('np.log(net_job + 1) ~ has_disablity  + EntityEffects', data=panel2019_2020).fit()#.fit(cov_type='clustered', cluster_entity=True, group_debias=False)
print(model1.summary)

                           PanelOLS Estimation Summary                           
Dep. Variable:     np.log(net_job + 1)   R-squared:                        0.0014
Estimator:                    PanelOLS   R-squared (Between):             -0.0088
No. Observations:                 4772   R-squared (Within):               0.0014
Date:                 Sat, Apr 25 2026   R-squared (Overall):             -0.0085
Time:                         18:21:33   Log-likelihood                   -9067.4
Cov. Estimator:             Unadjusted                                           
                                         F-statistic:                      3.3730
Entities:                         2386   P-value                           0.0664
Avg Obs:                        2.0000   Distribution:                  F(1,2385)
Min Obs:                        2.0000                                           
Max Obs:                        2.0000   F-statistic (robust):             3.3730
                

In [230]:
# 
model2 = PanelOLS.from_formula('np.log(net_job + 1) ~ has_disablity + disease_eye + chronic_spinal + disease_neuro + EntityEffects', data=panel2019_2020).fit()#.fit(cov_type='clustered', cluster_entity=True, group_debias=False)
print(model2.summary)

                           PanelOLS Estimation Summary                           
Dep. Variable:     np.log(net_job + 1)   R-squared:                        0.0024
Estimator:                    PanelOLS   R-squared (Between):              0.0022
No. Observations:                 4772   R-squared (Within):               0.0024
Date:                 Sat, Apr 25 2026   R-squared (Overall):              0.0022
Time:                         18:21:34   Log-likelihood                   -9065.0
Cov. Estimator:             Unadjusted                                           
                                         F-statistic:                      1.4409
Entities:                         2386   P-value                           0.2179
Avg Obs:                        2.0000   Distribution:                  F(4,2382)
Min Obs:                        2.0000                                           
Max Obs:                        2.0000   F-statistic (robust):             1.4409
                

In [231]:
model3 = PanelOLS.from_formula('net_job ~ has_disablity + EntityEffects', data=panel2019_2020).fit()#.fit(cov_type='clustered', cluster_entity=True, group_debias=False)
print(model3.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                net_job   R-squared:                        0.0001
Estimator:                   PanelOLS   R-squared (Between):             -0.0028
No. Observations:                4772   R-squared (Within):               0.0001
Date:                Sat, Apr 25 2026   R-squared (Overall):             -0.0026
Time:                        18:21:34   Log-likelihood                -5.109e+04
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      0.2625
Entities:                        2386   P-value                           0.6085
Avg Obs:                       2.0000   Distribution:                  F(1,2385)
Min Obs:                       2.0000                                           
Max Obs:                       2.0000   F-statistic (robust):             0.2625
                            

In [232]:
model4 = PanelOLS.from_formula('net_job ~ has_disablity + disease_eye + chronic_spinal + disease_neuro + EntityEffects', data=panel2019_2020).fit()#.fit(cov_type='clustered', cluster_entity=True, group_debias=False)
print(model4.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                net_job   R-squared:                        0.0005
Estimator:                   PanelOLS   R-squared (Between):              0.0068
No. Observations:                4772   R-squared (Within):               0.0005
Date:                Sat, Apr 25 2026   R-squared (Overall):              0.0063
Time:                        18:21:35   Log-likelihood                -5.109e+04
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      0.3251
Entities:                        2386   P-value                           0.8613
Avg Obs:                       2.0000   Distribution:                  F(4,2382)
Min Obs:                       2.0000                                           
Max Obs:                       2.0000   F-statistic (robust):             0.3251
                            

In [236]:
model5 = PanelOLS.from_formula('np.log(net_job + 1) ~ has_disability_common + EntityEffects + TimeEffects', data=panel2019_2020).fit(cov_type='clustered', cluster_entity=True, group_debias=False)
print(model5.summary)

AbsorbingEffectError: 
The model cannot be estimated. The included effects have fully absorbed
one or more of the variables. This occurs when one or more of the dependent
variable is perfectly explained using the effects included in the model.

The following variables or variable combinations have been fully absorbed
or have become perfectly collinear after effects are removed:

          has_disability_common

Set drop_absorbed=True to automatically drop absorbed variables.


In [235]:
model6 = PanelOLS.from_formula('net_job ~ has_disability_common + EntityEffects', data=panel2019_2020).fit()#.fit(cov_type='clustered', cluster_entity=True, group_debias=False)
print(model6.summary)

AbsorbingEffectError: 
The model cannot be estimated. The included effects have fully absorbed
one or more of the variables. This occurs when one or more of the dependent
variable is perfectly explained using the effects included in the model.

The following variables or variable combinations have been fully absorbed
or have become perfectly collinear after effects are removed:

          has_disability_common

Set drop_absorbed=True to automatically drop absorbed variables.


In [183]:
print(compare({'model 1':model1, "model 1 ext": model2, "model 2": model3, 'model 2 ext':model4}, stars=True))

                                           Model Comparison                                          
                                        model 1             model 1 ext        model 2    model 2 ext
-----------------------------------------------------------------------------------------------------
Dep. Variable               np.log(net_job + 1)     np.log(net_job + 1)        net_job        net_job
Estimator                              PanelOLS                PanelOLS       PanelOLS       PanelOLS
No. Observations                           4772                    4772           4772           4772
Cov. Est.                            Unadjusted              Unadjusted     Unadjusted     Unadjusted
R-squared                                0.0014                  0.0024         0.0001         0.0005
R-Squared (Within)                       0.0014                  0.0024         0.0001         0.0005
R-Squared (Between)                     -0.0088                  0.0022        -0.

# classical notation try

In [240]:
panel2019_2020 = (
    working_data
        # .filter(pl.col())
        .filter(~pl.col('idind').is_in(ids_to_exclude_0 + ids_to_exclude_1))
        #.with_columns(pl.col('year').cast(pl.Int32).cast(pl.String).str.to_date("%Y"))
        .with_columns(
            period = pl.when(pl.col('year') == 2020).then(1).otherwise(0)
        )
).to_pandas()
panel2019_2020

,idind,year,h7_2,h7_1,origsm,region,psu,status,age,educ,...,is_employed_no_disablity,net_job,educ_level,is_married,period_relevance,is_town,is_female,has_disability_2019,has_disability_common,period
0,60.0,2019.0,November,4.0,"Yes, it is an adress from the representative s...",Leningrad Oblast: Volosovkij Rajon,Volosovkij Rajon: Leningradskaya Oblast,PGT,51.0,10 and more grades of school & any professiona...,...,1,18000.0,higher,1,0,1,1,0,0,0
1,60.0,2020.0,September,30.0,"Yes, it is an adress from the representative s...",Leningrad Oblast: Volosovkij Rajon,Volosovkij Rajon: Leningradskaya Oblast,PGT,52.0,10 and more grades of school & any professiona...,...,1,30000.0,higher,1,1,1,1,0,0,1
2,125.0,2019.0,November,13.0,"Yes, it is an adress from the representative s...",Leningrad Oblast: Volosovkij Rajon,Volosovkij Rajon: Leningradskaya Oblast,rural,56.0,Secondary School Diploma,...,1,20000.0,common,1,0,0,1,0,0,0
3,125.0,2020.0,November,9.0,"Yes, it is an adress from the representative s...",Leningrad Oblast: Volosovkij Rajon,Volosovkij Rajon: Leningradskaya Oblast,rural,57.0,Secondary School Diploma,...,1,29500.0,common,1,1,0,1,0,0,1
4,126.0,2019.0,November,13.0,"Yes, it is an adress from the representative s...",Leningrad Oblast: Volosovkij Rajon,Volosovkij Rajon: Leningradskaya Oblast,rural,50.5,10 and more grades of school & any professiona...,...,1,20000.0,higher,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4767,59275.0,2020.0,September,29.0,"Yes, it is an adress from the representative s...",Novosibirskaya Oblast: Berdsk City & Raion (20...,"Tumenskaya Obl, Surgutskij R-n: Khanty-Mansi A...",town,25.5,Secondary School Diploma,...,1,27000.0,common,0,1,1,0,0,0,1
4768,59277.0,2019.0,November,21.0,"Yes, it is an adress from the representative s...",Moscow City,Moscow City,oblastnoy center,32.5,"Institute, University, Academy Diploma",...,1,40000.0,university,0,0,1,1,0,0,0
4769,59277.0,2020.0,September,12.0,"Yes, it is an adress from the representative s...",Moscow City,Moscow City,oblastnoy center,33.5,"Institute, University, Academy Diploma",...,1,40000.0,university,0,1,1,1,0,0,1
4770,59285.0,2019.0,November,21.0,"Yes, it is an adress from the representative s...",Moscow City,Moscow City,oblastnoy center,41.0,"Graduate school, residency without diploma",...,1,130000.0,common,1,0,1,0,0,0,0


In [242]:
ols1 = smf.ols('net_job ~ has_disability_common + period + I(has_disability_common * period)', panel2019_2020).fit()
print(ols1.summary2())

                              Results: Ordinary least squares
Model:                      OLS                      Adj. R-squared:             0.002      
Dependent Variable:         net_job                  AIC:                        110145.7709
Date:                       2026-04-25 18:29         BIC:                        110171.6530
No. Observations:           4772                     Log-Likelihood:             -55069.    
Df Model:                   3                        F-statistic:                3.516      
Df Residuals:               4768                     Prob (F-statistic):         0.0145     
R-squared:                  0.002                    Scale:                      6.1860e+08 
--------------------------------------------------------------------------------------------
                                    Coef.     Std.Err.    t    P>|t|     [0.025     0.975]  
--------------------------------------------------------------------------------------------
Intercep

In [243]:
ols2 = smf.ols('np.log(net_job + 1) ~ has_disability_common + period + I(has_disability_common * period)', panel2019_2020).fit()
print(ols2.summary2())

                         Results: Ordinary least squares
Model:                   OLS                    Adj. R-squared:        0.001     
Dependent Variable:      np.log(net_job + 1)    AIC:                   23349.4803
Date:                    2026-04-25 18:30       BIC:                   23375.3624
No. Observations:        4772                   Log-Likelihood:        -11671.   
Df Model:                3                      F-statistic:           2.134     
Df Residuals:            4768                   Prob (F-statistic):    0.0938    
R-squared:               0.001                  Scale:                 7.8014    
---------------------------------------------------------------------------------
                                   Coef.  Std.Err.    t     P>|t|   [0.025 0.975]
---------------------------------------------------------------------------------
Intercept                          9.4869   0.0579 163.8463 0.0000  9.3734 9.6004
has_disability_common             -0.5954